# Day 2, Session 2 -- Demos: LangChain Development & Tool Integration

This notebook contains all five instructor-led demos from Session 2. Run each cell, observe the output, and make sure you understand the pattern before moving on.

**Engineering context:** You are the engineer building LLM-powered applications with LangChain. Consultants are your users -- they need reusable analysis templates, composable pipelines, custom analytical tools, and knowledge retrieval systems they can query in natural language.

In [2]:
!pip install -q langchain langchain-openai langchain-core langchain-text-splitters python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Setup

In [3]:
from dotenv import load_dotenv
load_dotenv()

import json
import os

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports successful!")

All imports successful!


---
## Demo 1: LangChain Basics -- ChatModels and PromptTemplates

ChatModels wrap LLM APIs into a unified interface. PromptTemplates let you parameterize prompts with variables so they become reusable across different inputs.

**Scenario:** An engagement team needs to quickly generate structured analyses across different industries and strategy frameworks. PromptTemplates let us build reusable "analysis templates" that any consultant can invoke with different client parameters.

In [4]:
# Demo 1 - LangChain Basics: ChatModels and PromptTemplates

# Step 1: Initialize a ChatModel
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

# Step 2: Simple invocation with a string
response = llm.invoke("What is the MECE framework in one sentence?")
print("Simple invocation:")
print(f"  Type: {type(response)}")
print(f"  Content: {response.content}")

# Step 3: Invocation with message objects
messages = [
    SystemMessage(content="You are a senior partner. Be concise and strategic."),
    HumanMessage(content="What is a value chain analysis?")
]
response = llm.invoke(messages)
print(f"\nWith messages:\n  {response.content}")

# Step 4: PromptTemplates for reusable consulting prompts
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a consultant specializing in {domain}. Answer in {style} style."),
    ("human", "{question}")
])

formatted = prompt.format_messages(
    domain="growth strategy and market entry",
    style="executive briefing",
    question="What are the key drivers of growth in the EV battery market?"
)
print(f"\nFormatted prompt messages: {len(formatted)}")
for msg in formatted:
    print(f"  [{msg.type}]: {msg.content[:80]}...")

response = llm.invoke(formatted)
print(f"\nResponse: {response.content[:200]}...")

Simple invocation:
  Type: <class 'langchain_core.messages.ai.AIMessage'>
  Content: The MECE framework, which stands for "Mutually Exclusive, Collectively Exhaustive," is a problem-solving approach used to organize information and ideas in a way that ensures no overlaps (mutually exclusive) and that all possibilities are covered (collectively exhaustive).

With messages:
  Value chain analysis is a strategic tool used to identify and evaluate the activities within an organization that create value for customers. It breaks down the company's operations into primary and support activities, allowing businesses to understand how each contributes to competitive advantage and profitability. By analyzing these activities, companies can identify inefficiencies, optimize processes, and enhance value delivery, ultimately improving their market position.

Formatted prompt messages: 2
  [system]: You are a consultant specializing in growth strategy and market entry. Answer in...
  [human]: What a

---
## Demo 2: Building Chains with LCEL (Pipe Operator)

LCEL (LangChain Expression Language) uses the `|` operator to compose components into chains. The pattern is: `prompt | model | parser`. Each component implements the Runnable interface, so they can be freely combined.

**Scenario:** Engagement teams need multi-step analysis pipelines: research a market, analyze competitive dynamics, then produce an executive summary. LCEL lets us chain these steps cleanly -- mirroring how a real consulting workflow flows from data gathering through synthesis to recommendation.

In [5]:
# Demo 2 - Building Chains with LCEL

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

# Step 1: Simple chain: prompt -> LLM -> string output
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strategy consultant. Provide clear, structured insights."),
    ("human", "Explain {concept} in exactly {num_sentences} sentences, using a business consulting context.")
])

chain = prompt | llm | StrOutputParser()

result = chain.invoke({"concept": "Porter's Five Forces", "num_sentences": "3"})
print("Simple chain result:")
print(result)

# Step 2: Chain with JSON output
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output a JSON object with keys: framework_name, definition, consulting_use_case, complexity (1-10)."),
    ("human", "Describe the consulting framework: {concept}")
])

json_chain = json_prompt | llm | JsonOutputParser()

result = json_chain.invoke({"concept": "MECE principle"})
print(f"\nJSON chain result (type={type(result).__name__}):")
print(json.dumps(result, indent=2))

# Step 3: Multi-step chains (research -> executive summary -> translation for global offices)
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this market research finding in one executive-ready sentence: {text}"
)
translate_prompt = ChatPromptTemplate.from_template(
    "Translate this executive summary to French for our Paris office: {text}"
)

summary_chain = summarize_prompt | llm | StrOutputParser()
translate_chain = translate_prompt | llm | StrOutputParser()

text = "The global electric vehicle battery market is projected to grow at a CAGR of 19.2% from 2024 to 2030. Key growth drivers include government incentives, declining lithium-ion costs, and rising consumer demand for sustainable transportation. Asia-Pacific dominates with 68% market share, led by CATL and BYD."

summary = summary_chain.invoke({"text": text})
print(f"\nExecutive Summary: {summary}")

translation = translate_chain.invoke({"text": summary})
print(f"French (for Paris office): {translation}")

Simple chain result:
Porter's Five Forces framework is a strategic tool used to analyze the competitive environment of an industry by examining five key factors: the threat of new entrants, the bargaining power of suppliers, the bargaining power of buyers, the threat of substitute products, and the intensity of competitive rivalry. By assessing these forces, businesses can identify potential challenges and opportunities within their market, enabling them to develop strategies that enhance their competitive advantage. Ultimately, this analysis helps organizations make informed decisions regarding market entry, pricing strategies, and resource allocation.

JSON chain result (type=dict):
{
  "framework_name": "MECE Principle",
  "definition": "MECE stands for 'Mutually Exclusive, Collectively Exhaustive.' It is a problem-solving framework used in consulting to ensure that information is organized in a way that avoids overlap (mutually exclusive) and covers all possible options (collective

---
## Demo 3: Creating and Using Custom Tools

Tools let LLMs interact with the real world. In LangChain, use the `@tool` decorator to turn any Python function into a tool that an LLM can discover and invoke. The docstring becomes the tool's description.

**Scenario:** Consultants rely on specialized analytical tools daily -- market sizing calculators, benchmark databases, financial ratio analyzers. Here we build custom tools that mirror these consulting utilities and bind them to an LLM so it can decide which tool to use based on a client question.

In [6]:
# Demo 3 - Creating and Using Custom Tools

# Step 1: Define custom consulting tools
@tool
def market_sizing_tool(population: float, adoption_rate: float, avg_revenue_per_user: float) -> dict:
    """Estimate total addressable market (TAM) given population, adoption rate (0-1), and average revenue per user."""
    tam = population * adoption_rate * avg_revenue_per_user
    return {
        "total_addressable_market": tam,
        "population": population,
        "adoption_rate_pct": f"{adoption_rate * 100:.1f}%",
        "arpu": avg_revenue_per_user,
        "formatted_tam": f"${tam:,.0f}"
    }

@tool
def benchmark_lookup_tool(metric: str) -> str:
    """Look up industry benchmark values for common consulting metrics."""
    benchmarks = {
        "saas_gross_margin": "70-85% gross margin is typical for SaaS companies",
        "retail_operating_margin": "3-5% operating margin is typical for retail",
        "pharma_r_and_d_spend": "15-25% of revenue is typical pharma R&D spend",
        "tech_revenue_growth": "20-40% YoY revenue growth for high-growth tech",
        "healthcare_ebitda_margin": "15-25% EBITDA margin for healthcare services",
    }
    return benchmarks.get(metric.lower(), f"No benchmark found for '{metric}'. Available: {list(benchmarks.keys())}")

@tool
def financial_ratio_tool(revenue: float, costs: float, assets: float) -> dict:
    """Calculate key financial ratios used in consulting engagements."""
    gross_margin = (revenue - costs) / revenue if revenue else 0
    asset_turnover = revenue / assets if assets else 0
    return {
        "gross_margin": f"{gross_margin:.1%}",
        "asset_turnover": f"{asset_turnover:.2f}x",
        "profit": f"${revenue - costs:,.0f}"
    }

# Step 2: Inspect tool metadata
print("Tool: market_sizing_tool")
print(f"  Name: {market_sizing_tool.name}")
print(f"  Description: {market_sizing_tool.description}")

# Step 3: Invoke tools directly
print(f"\nmarket_sizing_tool (US EV market): {market_sizing_tool.invoke({'population': 330_000_000, 'adoption_rate': 0.08, 'avg_revenue_per_user': 45000})}")
print(f"benchmark_lookup_tool: {benchmark_lookup_tool.invoke('saas_gross_margin')}")
print(f"financial_ratio_tool: {financial_ratio_tool.invoke({'revenue': 50_000_000, 'costs': 32_000_000, 'assets': 80_000_000})}")

# Step 4: Bind tools to an LLM
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
llm_with_tools = llm.bind_tools([market_sizing_tool, benchmark_lookup_tool, financial_ratio_tool])

response = llm_with_tools.invoke("What is the typical gross margin for a SaaS company?")
print(f"\nLLM response with tools:")
print(f"  Content: {response.content}")
print(f"  Tool calls: {response.tool_calls}")

Tool: market_sizing_tool
  Name: market_sizing_tool
  Description: Estimate total addressable market (TAM) given population, adoption rate (0-1), and average revenue per user.

market_sizing_tool (US EV market): {'total_addressable_market': 1188000000000.0, 'population': 330000000.0, 'adoption_rate_pct': '8.0%', 'arpu': 45000.0, 'formatted_tam': '$1,188,000,000,000'}
benchmark_lookup_tool: 70-85% gross margin is typical for SaaS companies
financial_ratio_tool: {'gross_margin': '36.0%', 'asset_turnover': '0.62x', 'profit': '$18,000,000'}

LLM response with tools:
  Content: 
  Tool calls: [{'name': 'benchmark_lookup_tool', 'args': {'metric': 'gross margin for SaaS companies'}, 'id': 'call_wEEPpyTkV4ypsNC8t628STyk', 'type': 'tool_call'}]


---
## Demo 4: Document Loading and Text Splitting

RAG starts with loading documents and splitting them into manageable chunks. The `RecursiveCharacterTextSplitter` splits at natural boundaries (paragraphs, sentences) and maintains overlap between chunks so context is not lost at boundaries.

**Scenario:** Consulting firms maintain vast knowledge bases -- industry reports, client case studies, best practice documents. Before we can retrieve relevant insights, we need to split these documents into chunks that fit within LLM context windows while preserving analytical coherence.

In [7]:
# Demo 4 - Document Loading and Text Splitting

# Step 1: Create sample consulting documents
documents = [
    Document(
        page_content="""McKinsey Global Institute research shows that generative AI could add $2.6 to $4.4 trillion
annually to the global economy across 63 use cases. The banking, high-tech, and life sciences
sectors stand to benefit most, with potential productivity gains of 3-5% of total industry revenue.
Organizations that move quickly on adoption will capture disproportionate value.""",
        metadata={"source": "mgi_genai_report.pdf", "chapter": "Executive Summary"}
    ),
    Document(
        page_content="""Post-merger integration (PMI) remains the most critical phase of any M&A transaction.
McKinsey research indicates that 70% of mergers fail to achieve expected synergies, primarily due to
cultural misalignment and slow integration execution. Best-practice PMI follows a 100-day plan
covering Day 1 readiness, synergy capture, operating model design, and cultural integration.""",
        metadata={"source": "mckinsey_ma_playbook.pdf", "chapter": "Post-Merger Integration"}
    )
]

print(f"Loaded {len(documents)} consulting documents")
for doc in documents:
    print(f"  Source: {doc.metadata['source']} | Chapter: {doc.metadata['chapter']} | Length: {len(doc.page_content)} chars")

# Step 2: Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "200")),
    chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)
print(f"\nSplit into {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
    print(f"\n  Chunk {i+1} ({len(chunk.page_content)} chars) [from {chunk.metadata['source']}]:")
    print(f"    {chunk.page_content[:100]}...")

# Step 3: Show how overlap works
print("\nOverlap demonstration:")
print(f"  End of chunk 1  : ...{chunks[0].page_content[-40:]}")
if len(chunks) > 1:
    print(f"  Start of chunk 2: {chunks[1].page_content[:40]}...")

Loaded 2 consulting documents
  Source: mgi_genai_report.pdf | Chapter: Executive Summary | Length: 366 chars
  Source: mckinsey_ma_playbook.pdf | Chapter: Post-Merger Integration | Length: 374 chars

Split into 4 chunks:

  Chunk 1 (285 chars) [from mgi_genai_report.pdf]:
    McKinsey Global Institute research shows that generative AI could add $2.6 to $4.4 trillion
annually...

  Chunk 2 (80 chars) [from mgi_genai_report.pdf]:
    Organizations that move quickly on adoption will capture disproportionate value....

  Chunk 3 (281 chars) [from mckinsey_ma_playbook.pdf]:
    Post-merger integration (PMI) remains the most critical phase of any M&A transaction.
McKinsey resea...

  Chunk 4 (92 chars) [from mckinsey_ma_playbook.pdf]:
    covering Day 1 readiness, synergy capture, operating model design, and cultural integration....

Overlap demonstration:
  End of chunk 1  : ...gains of 3-5% of total industry revenue.
  Start of chunk 2: Organizations that move quickly on adopt...


---
## Demo 5: Building a Simple RAG Pipeline

This demo puts it all together: a knowledge base of text chunks, a simple retrieval function, and an LCEL chain that generates answers grounded in retrieved context.

**Scenario:** Imagine a knowledge management system where consultants can query the firm's repository of industry insights, frameworks, and engagement best practices. The RAG pipeline retrieves the most relevant knowledge snippets and generates a grounded, evidence-based answer -- just like a seasoned partner drawing on decades of firm knowledge.

In [8]:
# Demo 5 - Building a Simple RAG Pipeline

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# Step 1: Consulting knowledge base
knowledge_base = [
    "The MECE framework (Mutually Exclusive, Collectively Exhaustive) is a foundational structuring principle. It ensures problem decomposition has no overlaps and no gaps, enabling rigorous hypothesis-driven analysis.",
    "The Three Horizons of Growth framework helps companies allocate resources: Horizon 1 (core business), Horizon 2 (emerging opportunities), Horizon 3 (transformational initiatives).",
    "Value chain analysis, developed by Michael Porter, maps the primary and support activities that create value for customers. Consultants use it to identify cost reduction and differentiation opportunities.",
    "The 7S Framework examines seven interdependent elements: Strategy, Structure, Systems, Shared Values, Style, Staff, and Skills. It is used for organizational effectiveness assessments.",
    "Digital transformation engagements typically follow a 'rewire' approach: reimagine the business model, rebuild the technology foundation, and rewire the organization for speed and agility.",
    "Post-merger integration (PMI) success depends on capturing synergies within the first 100 days. The PMI approach covers Day 1 readiness, clean rooms, synergy tracking, and cultural integration.",
]

# Step 2: Simple keyword-based retrieval
def simple_retrieve(query, k=2):
    scored = []
    query_words = set(query.lower().split())
    for chunk in knowledge_base:
        chunk_words = set(chunk.lower().split())
        overlap = len(query_words & chunk_words)
        scored.append((overlap, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [chunk for _, chunk in scored[:k]]

# Step 3: RAG chain
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a knowledge management assistant. Answer based ONLY on the context from the knowledge base. If not found, say 'This topic is not covered in the current knowledge base.'\n\nContext:\n{context}"),
    ("human", "{question}")
])

rag_chain = rag_prompt | llm | StrOutputParser()

# Step 4: Run consulting queries
questions = [
    "What is the MECE framework?",
    "How does post-merger integration work?",
    "What is the capital asset pricing model?"
]

for question in questions:
    retrieved = simple_retrieve(question)
    context = "\n".join(f"- {chunk}" for chunk in retrieved)
    answer = rag_chain.invoke({"context": context, "question": question})
    print(f"Q: {question}")
    print(f"A: {answer}\n")

Q: What is the MECE framework?
A: The MECE framework (Mutually Exclusive, Collectively Exhaustive) is a foundational structuring principle that ensures problem decomposition has no overlaps and no gaps. This enables rigorous hypothesis-driven analysis.

Q: How does post-merger integration work?
A: Post-merger integration (PMI) works by focusing on capturing synergies within the first 100 days following a merger. The PMI approach includes several key components: 

1. **Day 1 Readiness**: Preparing for the immediate transition on the first day after the merger.
2. **Clean Rooms**: Establishing secure environments for sensitive information to be shared and analyzed.
3. **Synergy Tracking**: Monitoring and measuring the synergies that are expected to be realized from the merger.
4. **Cultural Integration**: Addressing the integration of different organizational cultures to ensure a smooth transition.

Successful PMI relies on effectively managing these elements to achieve the desired outco

---
## Summary

In this demo notebook you saw five core LangChain building blocks:

1. **ChatModels & PromptTemplates** -- Unified LLM interface and reusable, parameterized prompt templates.
2. **LCEL Chains** -- The pipe operator (`|`) composes prompt, model, and parser into clean pipelines.
3. **Custom Tools** -- The `@tool` decorator turns Python functions into tools LLMs can discover and invoke.
4. **Document Loading & Splitting** -- Preparing knowledge base documents for retrieval by splitting into overlapping chunks.
5. **RAG Pipelines** -- Combining retrieval with generation to ground answers in external knowledge.

**Next:** Open `D2S2_exercises.ipynb` to practice creating custom consulting tools and binding them to an LLM.